In [1]:
# So, let us dive right in. For convenience, let's begin by enabling
# automatic reloading of modules when they change.
%load_ext autoreload
%autoreload 2

In [3]:
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
# Packages for the simple design
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from collections import OrderedDict
design = designs.DesignPlanar()
gui = MetalGUI(design)
design.overwrite_enabled = True


03:31PM 27s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


In [2]:
design.chips.main.size.size_x = '5mm'
design.chips.main.size.size_y = '5mm'
design.chips.main.size.size_z = '-380um'
design.chips.main.size.sample_holder_top='2mm'
design.chips.main.size.sample_holder_bottom='380um'


In [ ]:
try:
    gui.main_window.force_close = True
    gui.main_window.close()
except (NameError, AttributeError):
    pass
  
OTG1 = OpenToGround(design,name='OTG1'
                 ,options={'pos_x':'-1.1mm','pos_y':'-1.5mm','orientation':'0','width':'15.5um', 'gap':'9um', 'termination_gap': '9um' })


LP1 = LaunchpadWirebondDriven(design,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.0mm', 'pos_y': '-1.25mm','pad_width':'180um','pad_height':'180um','pad_gap':'140um', 'trace_width':'15.5um' , 'trace_gap': '9um'},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )




LP2 = LaunchpadWirebondDriven(design,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.0mm', 'pos_y': '1.25mm','pad_width':'180um','pad_height':'180um','pad_gap':'140um', 'trace_width':'15.5um' , 'trace_gap': '9um'},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

q_read1 = CoupledLineTee(design,'Q_Read1', options=dict(pos_x = '-1.775mm', pos_y = '-0.425mm',
                                                        prime_width='15.5um',
                                                        prime_gap='9um',
                                                        second_width='15.5um',
                                                        second_gap='9um',
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '90',
                                                        coupling_space = '4.5um',                                                         
                                                        coupling_length = '300um',
                                                        open_termination = False))



seg1 = RoutePathfinder(design, 'seg1', options = dict(chip='main', 
                                                      trace_width ='15.5um',
                                                      trace_gap ='9um',
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(
                                                        end_straight='0.2mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='Q_Read1',
                                                                pin='prime_start'),
                                                       end_pin=dict(
                                                        component='LP1',
                                                        pin='tie')
                                                 )))



seg2 = RoutePathfinder(design, 'seg2', options = dict(chip='main', 
                                                      trace_width ='15.5um',
                                                      trace_gap ='9um',
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0.125mm',
                                                        end_straight='0.2mm',start_jogged_extension = OrderedDict([(0,['R','0um'])])),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='Q_Read1',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


RR1 = RouteMeander(design, 'RR1',  Dict(
        trace_width ='15.5um',
        trace_gap ='9um',
        total_length='4mm',
        hfss_wire_bonds = False,
        fillet='50 um',
        lead = dict(start_straight='300um',end_straight='50um'),
        meander= dict(spacing='150um',asymmetry='0um'),
        pin_inputs=Dict(
            start_pin=Dict(component='Q_Read1', pin='second_end'),
            end_pin=Dict(component='OTG1', pin='open')), ))

gui.rebuild()
gui.autoscale()